# Synthetic multivariable workflow

This notebook uses simulated data to demonstrate analysis of a bounded outcome (0–1). It does not use private records, submissions, manuscript text, or original results.

The workflow includes categorical and numeric predictors, column standardization and basic validation, category counts and cross-tabulations, regression, and table export to a configurable directory.

All coefficients and p-values are simulation outputs and have no substantive interpretation.

In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

OUTPUT_DIR = Path(
    os.environ.get("RPA_TOOLKIT_OUTPUT_DIR", "synthetic_examples/outputs")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be written to: {OUTPUT_DIR.resolve()}")

## Synthetic data

Generate a bounded score on a 0–100 scale and convert it to a proportion for modeling. Predictors partially overlap so cross-tabs remain informative without reproducing the structure of an original corpus.

In [ ]:
def make_synthetic_dataset(n: int = 600, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    language = rng.choice(["Arabic", "English", "Code"], size=n, p=[0.32, 0.56, 0.12])

    assignment_map = {
        "Arabic": ["Article", "Reflection"],
        "English": ["Lab Report", "Project Report", "Case Study", "Research Summary"],
        "Code": ["Coding Project", "Technical Notebook"],
    }
    major_map = {
        "Arabic": ["General", "Humanities"],
        "English": ["Medicine", "Mechanical Engineering", "Software Engineering"],
        "Code": ["Software Engineering", "Data Science"],
    }
    university_map = {
        "Arabic": ["University_A", "University_B"],
        "English": ["University_A", "University_C", "University_D"],
        "Code": ["University_C", "University_D"],
    }

    assignment_type = [rng.choice(assignment_map[x]) for x in language]
    major = [rng.choice(major_map[x]) for x in language]
    university = [rng.choice(university_map[x]) for x in language]
    year_grouping = rng.choice(["First Year", "Second Year"], size=n, p=[0.58, 0.42])

    word_count = np.clip(
        rng.normal(loc=950, scale=260, size=n).round(), 150, None
    ).astype(int)
    log_word_count = np.log1p(word_count)

    lang_effect = (
        pd.Series(language)
        .map({"Arabic": 0.10, "English": -0.02, "Code": 0.06})
        .to_numpy()
    )
    year_effect = (
        pd.Series(year_grouping)
        .map({"First Year": 0.04, "Second Year": -0.03})
        .to_numpy()
    )
    assign_effect = (
        pd.Series(assignment_type)
        .map(
            {
                "Article": 0.08,
                "Reflection": 0.03,
                "Lab Report": -0.04,
                "Project Report": 0.01,
                "Case Study": -0.01,
                "Research Summary": 0.02,
                "Coding Project": 0.05,
                "Technical Notebook": 0.00,
            }
        )
        .to_numpy()
    )

    linear = (
        -0.35
        + lang_effect
        + year_effect
        + assign_effect
        + 0.06 * (log_word_count - log_word_count.mean())
    )
    noise = rng.normal(0, 0.12, size=n)
    outcome_prop = 1 / (1 + np.exp(-(linear + noise)))
    outcome_100 = np.clip(outcome_prop * 100, 0, 100)

    df = pd.DataFrame(
        {
            "file_name": [f"synthetic_{i:04d}.txt" for i in range(1, n + 1)],
            "language": language,
            "university": university,
            "major": major,
            "assignment_type": assignment_type,
            "year_original": year_grouping,
            "word_count": word_count,
            "combined_ai_rate_100": outcome_100,
        }
    )
    return df


raw = make_synthetic_dataset()
print("Synthetic raw shape:", raw.shape)
display(raw.head())

## 2. Standardize columns and derive analysis fields

In [ ]:
def normalize_colname(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"[%()]", "", name)
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


df = raw.rename(columns={c: normalize_colname(c) for c in raw.columns}).copy()

required = [
    "combined_ai_rate_100",
    "language",
    "assignment_type",
    "word_count",
    "major",
    "university",
    "year_original",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in ["language", "assignment_type", "major", "university", "year_original"]:
    df[col] = df[col].astype(str).str.strip()

df["word_count"] = pd.to_numeric(df["word_count"], errors="coerce")
df["combined_ai_rate_100"] = pd.to_numeric(
    df["combined_ai_rate_100"], errors="coerce"
).clip(0, 100)
df = df.dropna(subset=required).copy()

df["combined_ai_rate_prop"] = df["combined_ai_rate_100"] / 100.0
df["log_word_count"] = np.log1p(df["word_count"])
df["year_grouping"] = df["year_original"]
df["assignment_context"] = df["language"] + " / " + df["assignment_type"]

print("Clean analytic shape:", df.shape)
print(
    "Outcome range:", df["combined_ai_rate_100"].min(), df["combined_ai_rate_100"].max()
)
display(df.head())

## 3. Descriptive counts and cross-tabulations

Summarize overlap among grouping variables before fitting the adjusted model.

In [ ]:
group_vars = [
    "language",
    "assignment_type",
    "assignment_context",
    "major",
    "university",
    "year_grouping",
]

count_frames = []
for var in group_vars:
    counts = (
        df[var].value_counts(dropna=False).rename_axis("category").reset_index(name="n")
    )
    counts.insert(0, "variable", var)
    counts["percent"] = counts["n"] / len(df) * 100
    count_frames.append(counts)

category_counts = pd.concat(count_frames, ignore_index=True)
display(category_counts)
category_counts.to_csv(OUTPUT_DIR / "category_counts.csv", index=False)

In [ ]:
def crosstab_with_test(
    data: pd.DataFrame, left: str, right: str
) -> tuple[pd.DataFrame, dict]:
    ct = pd.crosstab(data[left], data[right], dropna=False)
    chi2, p_value, dof, expected = chi2_contingency(ct)
    summary = {
        "left": left,
        "right": right,
        "chi2": chi2,
        "p_value": p_value,
        "dof": dof,
        "min_expected": float(np.min(expected)),
    }
    return ct, summary


pairwise = [
    ("language", "assignment_type"),
    ("language", "major"),
    ("language", "university"),
    ("language", "year_grouping"),
    ("assignment_type", "major"),
    ("assignment_type", "university"),
    ("assignment_type", "year_grouping"),
]

test_rows = []
for left, right in pairwise:
    ct, summary = crosstab_with_test(df, left, right)
    print(f"\nCrosstab: {left} by {right}")
    display(ct)
    ct.to_csv(OUTPUT_DIR / f"crosstab_{left}_by_{right}.csv")
    test_rows.append(summary)

crosstab_tests = pd.DataFrame(test_rows)
display(crosstab_tests)
crosstab_tests.to_csv(OUTPUT_DIR / "crosstab_tests.csv", index=False)

## 4. Fit an adjusted model

Fit a binomial GLM to the outcome on the 0–1 scale. Because this synthetic example uses a continuous proportion rather than grouped successes and failures, the model is used to demonstrate formula specification and coefficient extraction; it is not a general recommendation for bounded outcomes.

In [ ]:
formula = (
    "combined_ai_rate_prop ~ C(assignment_type) + C(language) + log_word_count + "
    "C(major) + C(university) + C(year_grouping)"
)

model = smf.glm(
    formula=formula,
    data=df,
    family=__import__("statsmodels.api").api.families.Binomial(),
)
result = model.fit()
print(result.summary())

In [ ]:
coef_table = pd.DataFrame(
    {
        "term": result.params.index,
        "coef": result.params.values,
        "std_err": result.bse.values,
        "z": result.tvalues.values,
        "p_value": result.pvalues.values,
    }
)

conf_int = result.conf_int()
coef_table["ci_lower"] = conf_int[0].values
coef_table["ci_upper"] = conf_int[1].values
display(coef_table)
coef_table.to_csv(OUTPUT_DIR / "glm_coefficients.csv", index=False)

## 5. Predicted values and grouped summaries

In [ ]:
df["predicted_prop"] = result.predict(df)
df["predicted_100"] = df["predicted_prop"] * 100

group_summary = (
    df.groupby(["language", "assignment_type"], dropna=False)
    .agg(
        n=("file_name", "size"),
        observed_mean_100=("combined_ai_rate_100", "mean"),
        predicted_mean_100=("predicted_100", "mean"),
        mean_word_count=("word_count", "mean"),
    )
    .reset_index()
    .sort_values(["language", "assignment_type"])
)

display(group_summary)
group_summary.to_csv(OUTPUT_DIR / "group_summary_language_assignment.csv", index=False)

## 6. Export an analysis-ready synthetic dataset

This export is safe to share because it is generated within the notebook and contains no private source records.

In [ ]:
# Columns to include in the exported CSV (subset of dataframe fields)
export_cols = [
    "file_name",
    "language",
    "university",
    "major",
    "assignment_type",
    "year_grouping",
    "word_count",
    "log_word_count",
    "combined_ai_rate_100",
    "combined_ai_rate_prop",
    "predicted_100",
]

# Write selected columns to a CSV for downstream analysis
df[export_cols].to_csv(OUTPUT_DIR / "synthetic_analysis_dataset.csv", index=False)
print("Wrote synthetic dataset and summary tables.")